In [1]:
# imports
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

In [2]:
# load, sort by time, and define splits
DATA_PATH = 'Data_Files/cleaned_dataset_per_device.csv'
SAVE_DIR = 'Data_Files'
TEST_SIZE = 0.2  # hold out the most recent 20%

df = pd.read_csv(DATA_PATH, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)

feature_names = [
    'distance', 'frequency', 'c_walls', 'w_walls', 'co2', 'humidity',
    'pm25', 'pressure', 'temperature', 'snr'
]
target_col = 'exp_pl'

split_idx = int(len(df) * (1 - TEST_SIZE))
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

X_train = train_df[feature_names].to_numpy()
y_train = train_df[target_col].to_numpy()
time_train = train_df['time'].to_numpy()

X_test = test_df[feature_names].to_numpy()
y_test = test_df[target_col].to_numpy()
time_test = test_df['time'].to_numpy()

print(f"Training samples: {len(train_df)}, Test samples: {len(test_df)}")
print(f"Train window: {train_df.time.min()} -> {train_df.time.max()}")
print(f"Test  window: {test_df.time.min()} -> {test_df.time.max()}")

Training samples: 1663627, Test samples: 415907
Train window: 2024-10-01 00:01:07.420593+00:00 -> 2025-08-12 17:18:53.293125+00:00
Test  window: 2025-08-12 17:19:02.126782+00:00 -> 2025-09-30 23:59:55.971870+00:00


In [3]:
# save the time-ordered splits (carry device_id through)

# metadata to carry through outputs
meta_cols = [c for c in ['device_id'] if c in df.columns]

os.makedirs(SAVE_DIR, exist_ok=True)

# base columns to carry forward (meta + features + PL + time)
base_cols = meta_cols + feature_names

train_df_out = train_df[base_cols].copy()
train_df_out['PL'] = y_train
train_df_out['time'] = time_train

test_df_out = test_df[base_cols].copy()
test_df_out['PL'] = y_test
test_df_out['time'] = time_test

train_df_out.to_csv(f"{SAVE_DIR}/train.csv", index=False)
test_df_out.to_csv(f"{SAVE_DIR}/test.csv", index=False)
print(f"Saved train.csv and test.csv to {SAVE_DIR}")

Saved train.csv and test.csv to Data_Files


In [ ]:
# time-series CV splits for model selection and OOF diagnostics (training set only)
N_SPLITS = 5
GAP = 0  # row gap between train/validation windows

if not train_df['time'].is_monotonic_increasing:
    raise ValueError('train_df must be sorted by time before creating time-series CV splits.')

if train_df['time'].max() >= test_df['time'].min():
    raise ValueError('Train/test split is not strictly time ordered.')

tscv = TimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)
cv_splits = list(tscv.split(train_df))

# Validation labels are metadata only. -1 marks the initial prefix that TimeSeriesSplit
# uses for training but never validates. For expanding-window CV train_time_series_cv_splits.npz is used.
fold_assignments = np.full(len(train_df), -1, dtype=int)
seen_val = np.zeros(len(train_df), dtype=bool)

split_arrays = {
    'n_splits': np.array(N_SPLITS, dtype=np.int64),
    'gap': np.array(GAP, dtype=np.int64),
}
audit_rows = []

for fold_num, (tr_idx, val_idx) in enumerate(cv_splits):
    if np.intersect1d(tr_idx, val_idx).size:
        raise ValueError(f'Fold {fold_num}: train/validation overlap detected.')

    if tr_idx.max() >= val_idx.min():
        raise ValueError(f'Fold {fold_num}: future rows appear in training.')

    row_gap = val_idx.min() - tr_idx.max() - 1
    if row_gap < GAP:
        raise ValueError(f'Fold {fold_num}: expected at least gap {GAP}, got {row_gap}.')

    train_end = train_df['time'].iloc[tr_idx].max()
    val_start = train_df['time'].iloc[val_idx].min()
    if train_end >= val_start:
        raise ValueError(f'Fold {fold_num}: time leakage detected.')

    if seen_val[val_idx].any():
        raise ValueError(f'Fold {fold_num}: validation rows reused across folds.')

    seen_val[val_idx] = True
    fold_assignments[val_idx] = fold_num

    split_arrays[f'fold_{fold_num}_train_idx'] = tr_idx.astype(np.int64)
    split_arrays[f'fold_{fold_num}_val_idx'] = val_idx.astype(np.int64)

    audit_rows.append({
        'fold': fold_num,
        'train_n': len(tr_idx),
        'val_n': len(val_idx),
        'gap_rows': row_gap,
        'train_start': train_df['time'].iloc[tr_idx].min(),
        'train_end': train_end,
        'val_start': val_start,
        'val_end': train_df['time'].iloc[val_idx].max(),
    })

cv_audit = pd.DataFrame(audit_rows)
display(cv_audit)

label_counts = pd.Series(fold_assignments[fold_assignments >= 0]).value_counts().sort_index()
expected_counts = cv_audit.set_index('fold')['val_n'].astype(int)
if not label_counts.astype(int).equals(expected_counts):
    raise ValueError('Fold-label sanity check failed: validation label counts do not match saved split sizes.')

if int(seen_val.sum()) != int(cv_audit['val_n'].sum()):
    raise ValueError('OOF coverage sanity check failed: seen validation rows do not match audit totals.')

if int((fold_assignments == -1).sum()) == 0:
    raise ValueError('Fold-label sanity check failed: expected an initial non-validation prefix marked -1.')


np.savez_compressed(f'{SAVE_DIR}/train_time_series_cv_splits.npz', **split_arrays)
np.save(f'{SAVE_DIR}/train_folds.npy', fold_assignments)
cv_audit.to_csv(f'{SAVE_DIR}/train_time_series_cv_audit.csv', index=False)

print(f"Saved expanding-window CV splits to {SAVE_DIR}/train_time_series_cv_splits.npz")
print(f"Saved validation-fold metadata to {SAVE_DIR}/train_folds.npy")
print(f"Saved CV audit to {SAVE_DIR}/train_time_series_cv_audit.csv")
print(f"OOF-valid rows: {seen_val.sum():,} / {len(train_df):,}")
print(f"Initial prefix rows without OOF prediction: {(fold_assignments == -1).sum():,}")
print('CV audit passed: each split trains strictly before its validation window.')

,fold,train_n,val_n,gap_rows,train_start,train_end,val_start,val_end
0,0,277272,277271,0,2024-10-01 00:01:07.420593+00:00,2024-11-22 16:14:10.475164+00:00,2024-11-22 16:14:36.579002+00:00,2025-01-12 19:00:02.336990+00:00
1,1,554543,277271,0,2024-10-01 00:01:07.420593+00:00,2025-01-12 19:00:02.336990+00:00,2025-01-12 19:00:12.120847+00:00,2025-03-16 11:49:03.885341+00:00
2,2,831814,277271,0,2024-10-01 00:01:07.420593+00:00,2025-03-16 11:49:03.885341+00:00,2025-03-16 11:49:27.286563+00:00,2025-05-07 02:59:11.265952+00:00
3,3,1109085,277271,0,2024-10-01 00:01:07.420593+00:00,2025-05-07 02:59:11.265952+00:00,2025-05-07 02:59:25.336805+00:00,2025-06-30 14:22:42.066099+00:00
4,4,1386356,277271,0,2024-10-01 00:01:07.420593+00:00,2025-06-30 14:22:42.066099+00:00,2025-06-30 14:22:54.211608+00:00,2025-08-12 17:18:53.293125+00:00


Saved expanding-window CV splits to Data_Files/train_time_series_cv_splits.npz
Saved validation-fold metadata to Data_Files/train_folds.npy
Saved CV audit to Data_Files/train_time_series_cv_audit.csv
OOF-valid rows: 1,386,355 / 1,663,627
Initial prefix rows without OOF prediction: 277,272
CV audit passed: each split trains strictly before its validation window.
